In [ ]:
# hide
from typing import Iterator
import numpy as np
import pyquist as pq
import matplotlib.pyplot as plt

In [ ]:
# hide
def iter_frames(audio: pq.Audio, N_H: int, N_F: int) -> Iterator[np.ndarray]:
    for start in range(0, len(audio) - N_F + 1, N_H):
        yield audio.samples[start:start + N_F]

In [ ]:
def rms(x: np.ndarray) -> float:
    """The root-mean-square amplitude of x."""
    return np.sqrt(np.mean(x ** 2))



def rms_envelope(audio: pq.Audio, N_H: int, N_F: int) -> np.ndarray:
    """The RMS amplitude of each frame of the audio (Chapter 10)."""
    return np.array([rms(frame) for frame in iter_frames(audio, N_H, N_F)])


f_s = 44100
n = np.arange(f_s)
peak = pq.helper.db_to_amplitude(-30.0)              # a gentle peak amplitude
click = np.zeros(f_s)
click[0] = peak
sine = peak * np.sin(2 * np.pi * 440 * n / f_s)
noise = peak * np.random.choice([-1.0, 1.0], f_s)   # white noise: each sample is -peak or peak
for name, x in [("click", click), ("sine", sine), ("noise", noise)]:
    print(f"{name}: peak {np.abs(x).max():.4f}, RMS {rms(x):.4f}")
    pq.play(pq.Audio(x, f_s))

beethoven = pq.Audio.from_file("../assets/beethoven.wav")
pq.play(beethoven, normalize=True)
N_H = beethoven.sample_rate // 100                  # 100 frames per second
level_db = pq.helper.amplitude_to_db(np.maximum(rms_envelope(beethoven, N_H, 4 * N_H), 1e-6))
plt.plot(np.arange(len(level_db)) * N_H / beethoven.sample_rate, level_db)
plt.xlabel("Time (s)")
plt.ylabel("RMS level (dB)")
plt.ylim(-80, 0)
plt.show()